In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [15]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhavi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

In [17]:
df = pd.read_csv('tweets.csv', encoding= 'ISO-8859-1')

In [18]:
df.head()

,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it. ;D"
0,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
1,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
2,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
3,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."
4,0,1467811372,Mon Apr 06 22:20:00 PDT 2009,NO_QUERY,joy_wolf,@Kwesidei not the whole crew


In [23]:
#Naming the columns and reading dataset again

col_names = ['target', 'id', 'date', 'flag', 'user', 'tweet']

In [24]:
tweet_data = pd.read_csv('tweets.csv', names=col_names, encoding= 'ISO-8859-1')

In [25]:
print("numbere of rows in the dataset are:",tweet_data.shape[0])
print("numbere of columns in the dataset are:",tweet_data.shape[1])

numbere of rows in the dataset are: 1600000
numbere of columns in the dataset are: 6


In [26]:
tweet_data.sample(5)

,target,id,date,flag,user,tweet
290797,0,1995469071,Mon Jun 01 13:32:22 PDT 2009,NO_QUERY,skeggiejohn,Time for bed - have to be up @ 2.30 am - long ...
1534938,4,2178886466,Mon Jun 15 08:20:43 PDT 2009,NO_QUERY,JacobEck,"@MeRide It's only 1 hour and 30 mins. Besides,..."
1345237,4,2043955973,Fri Jun 05 08:39:38 PDT 2009,NO_QUERY,FloTom,@alandavies1 You should have voted English Dem...
250462,0,1983224403,Sun May 31 12:44:45 PDT 2009,NO_QUERY,Melinasings,doesnt wanna write her exam tomorrow aaahhh h...
475728,0,2177691670,Mon Jun 15 06:32:30 PDT 2009,NO_QUERY,Kelly2686,I want to play burnout


In [65]:
tweet_data.isnull().sum()

target             0
id                 0
date               0
flag               0
user               0
tweet              0
stemmed_content    0
dtype: int64

In [66]:
tweet_data.describe()

,target,id
count,1600000.0,1.600000e+06
mean,0.5,1.998818e+09
std,0.5,1.935761e+08
min,0.0,1.467810e+09
25%,0.0,1.956916e+09
50%,0.5,2.002102e+09
75%,1.0,2.177059e+09
max,1.0,2.329206e+09


In [67]:
tweet_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1600000 entries, 0 to 1599999
Data columns (total 7 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   target           1600000 non-null  int64 
 1   id               1600000 non-null  int64 
 2   date             1600000 non-null  object
 3   flag             1600000 non-null  object
 4   user             1600000 non-null  object
 5   tweet            1600000 non-null  object
 6   stemmed_content  1600000 non-null  object
dtypes: int64(2), object(5)
memory usage: 85.4+ MB


In [68]:
tweet_data['target'].value_counts()

target
0    800000
1    800000
Name: count, dtype: int64

In [69]:
#Convert the target label  4 to 1 for better understanding
tweet_data.replace({'target':{4:1}}, inplace=True)

# 0 means negative and 1 means positive


In [145]:
#Stemming (It reduces word to its root word)
port_stem = PorterStemmer()


In [146]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-z]',' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stop_words = set(stopwords.words('english'))
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stop_words]
    stemmed_content = ' '.join(stemmed_content)
    
    return stemmed_content

In [148]:
tweet_data['stemmed_content'] = tweet_data['tweet'].apply(stemming)

In [111]:
import nltk
nltk.download('wordnet')
nltk.download('stopwords')


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\bhavi\AppData\Roaming\nltk_data...
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\bhavi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [112]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatizing(content):
    # Remove non-alphabetical characters
    lemmatized_content = re.sub('[^a-zA-Z]', ' ', content)
    
    # Convert to lowercase and split the content into words
    lemmatized_content = lemmatized_content.lower().split()
    
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    lemmatized_content = [lemmatizer.lemmatize(word) for word in lemmatized_content if word not in stop_words]
    
    # Join the words back into a string
    lemmatized_content = ' '.join(lemmatized_content)
    
    return lemmatized_content

# Apply the lemmatizing function to the tweet data
tweet_data['lemmatized_content'] = tweet_data['tweet'].apply(lemmatizing)


In [150]:
tweet_data.head()

,target,id,date,flag,user,tweet,stemmed_content,lemmatized_content
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",switchfoot http twitpic com zl awww bummer sho...,switchfoot http twitpic com zl awww bummer sho...
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...,upset updat facebook text might cri result sch...,upset update facebook texting might cry result...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...,kenichan dive mani time ball manag save rest g...,kenichan dived many time ball managed save res...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire,whole bodi feel itchi like fire,whole body feel itchy like fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all....",nationwideclass behav mad see,nationwideclass behaving mad see


In [151]:
X = tweet_data['stemmed_content'].values
y = tweet_data['target'].values

In [152]:
print(X)

['switchfoot http twitpic com zl awww bummer shoulda got david carr third day'
 'upset updat facebook text might cri result school today also blah'
 'kenichan dive mani time ball manag save rest go bound' ...
 'readi mojo makeov ask detail'
 'happi th birthday boo alll time tupac amaru shakur'
 'happi charitytuesday thenspcc sparkschar speakinguph h']


In [153]:
print(y)

[0 0 0 ... 1 1 1]


In [154]:
x = tweet_data['lemmatized_content'].values
Y = tweet_data['target'].values

In [155]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)

In [156]:
x_train, x_test, Y_train, Y_test = train_test_split(x, Y, test_size=0.2, random_state=5)

In [157]:
print(X.shape,X_train.shape,X_test.shape)

(1600000,) (1280000,) (320000,)


In [158]:
print(x.shape,x_train.shape,x_test.shape)

(1600000,) (1280000,) (320000,)


In [159]:
vectorizer = TfidfVectorizer()


In [161]:
X_train = vectorizer.fit_transform(X_train)

X_test = vectorizer.transform(X_test)

AttributeError: 'csr_matrix' object has no attribute 'lower'

In [162]:
print(X_train)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9416785 stored elements and shape (1280000, 483277)>
  Coords	Values
  (0, 434151)	0.42836201353563563
  (0, 8772)	0.32312461707333845
  (0, 177933)	0.32263359032529376
  (0, 187246)	0.16794483321395784
  (0, 131952)	0.19623135745745746
  (0, 430388)	0.20701024130205414
  (0, 409347)	0.35980917690240755
  (0, 465906)	0.23533014018460674
  (0, 395047)	0.2443692594950218
  (0, 294081)	0.1785903678560877
  (0, 159228)	0.14522995212621623
  (0, 256787)	0.2320307291580631
  (0, 373944)	0.20060663423106304
  (0, 432898)	0.18125670427093826
  (0, 473684)	0.27195182475839985
  (1, 468453)	0.42119148224996855
  (1, 158883)	0.6076539468962595
  (1, 169286)	0.3013176738798824
  (1, 224546)	0.3250456526856547
  (1, 107319)	0.3572950489573565
  (1, 337144)	0.35951028798882234
  (2, 378050)	0.7082090923679262
  (2, 140235)	0.5330611432148558
  (2, 426453)	0.46291003346424564
  (3, 183015)	0.5391033658032189
  :	:
  (1279994, 334724)	0.253

In [163]:
print(X_test)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 2274206 stored elements and shape (320000, 483277)>
  Coords	Values
  (0, 31145)	0.37492792227330896
  (0, 34810)	0.12148651326393398
  (0, 47074)	0.14457183094039672
  (0, 52715)	0.32069617803251965
  (0, 76433)	0.27851396856203675
  (0, 85172)	0.19081073466782728
  (0, 133082)	0.2249923163253429
  (0, 146088)	0.17711744022872347
  (0, 148519)	0.14084373324294105
  (0, 151160)	0.36182910749716335
  (0, 160160)	0.30710408306104187
  (0, 161753)	0.13641372544779082
  (0, 164161)	0.28328146562097195
  (0, 172250)	0.36555057446029376
  (0, 404976)	0.2177099820850432
  (1, 32839)	0.27661901868863814
  (1, 88444)	0.34811276014357045
  (1, 154582)	0.3970601387288004
  (1, 155151)	0.429586989931306
  (1, 254856)	0.1780944048828739
  (1, 294081)	0.21192228247254766
  (1, 309644)	0.3592415640604666
  (1, 404367)	0.2676874635657963
  (1, 455144)	0.3093917164984753
  (1, 467395)	0.2950164477510593
  :	:
  (319997, 312031)	0.38430831752

In [164]:
#LogisticRegression model training
model = LogisticRegression(max_iter=1000)

In [165]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [166]:
#accuracy 
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(y_train,X_train_prediction)


In [ ]:
#accuracy 
x_train_prediction = model.predict(x_train)
small_training_data_accuracy  =accuracy_score(Y_train,x_train_prediction)


In [101]:
print("accuracy sore on the training data:", training_data_accuracy)

accuracy sore on the training data: 0.799371875


In [102]:
#accuracy 
X_test_prediction = model.predict(X_test)
testing_data_accuracy  =accuracy_score(y_test,X_test_prediction)


In [103]:
print("Accuracy score on test dataset:", testing_data_accuracy)

Accuracy score on test dataset: 0.77749375


In [104]:
import pickle

In [105]:
filename = 'trained_model.sav'
pickle.dump(model, open(filename, 'wb'))

In [108]:
filename = 'vectorizer.pkl'
with open(filename, 'wb') as file:
    pickle.dump(vectorizer, file)